# Simulation Implementation Notebook (Part-by-Part)

This notebook implements the plan in incremental parts.

## Planned Parts
1. **Part 1 (implemented now):** Data inventory and schema audit for Twitter + OpenAssistant
2. Part 2: Action labeling pipeline (LLM + QA sample)
3. Part 3: Persona feature table + GMM clustering
4. Part 4: RAG index build (turn-level + conversation-level)
5. Part 5: Simulator interface (`reset`, `step`) with state extraction
6. Part 6: Bandit baselines + PPO warm-start scaffold

This run focuses only on **Part 1**.

In [2]:
from pathlib import Path
import csv
import json
import pandas as pd

# Input dataset roots
ROOT = Path(r"B:\\College\\RL\\AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service")
TWITTER_ROOT = ROOT / "twitter"
OPENASSIST_ROOT = ROOT / "OpenAssistant Conversations Dataset"

# Output folder for Part 1 artifacts
OUT_DIR = ROOT / "Simulation" / "artifacts" / "part1_data_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TWITTER_ROOT, OPENASSIST_ROOT, OUT_DIR

(WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/twitter'),
 WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/OpenAssistant Conversations Dataset'),
 WindowsPath('B:/College/RL/AdaptiveBandit--Contextual-Bandits-for-Real-Time-Decision-Support-in-Customer-Service/Simulation/artifacts/part1_data_audit'))

In [3]:
def discover_csv_files(*roots):
    files = []
    for root in roots:
        if root.exists():
            files.extend(sorted(root.rglob("*.csv")))
    return files


def count_data_rows(csv_path: Path):
    # Fast line count minus header row; robust fallback if file is empty.
    total_lines = 0
    with csv_path.open("r", encoding="utf-8", errors="ignore") as f:
        for _ in f:
            total_lines += 1
    return max(total_lines - 1, 0)


def read_header(csv_path: Path):
    with csv_path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
        reader = csv.reader(f)
        try:
            return next(reader)
        except StopIteration:
            return []


def sample_missingness(csv_path: Path, sample_rows=50000):
    sample_df = pd.read_csv(csv_path, nrows=sample_rows, low_memory=False)
    miss = (sample_df.isna().mean() * 100).round(2)
    out = pd.DataFrame({
        "column": miss.index,
        "missing_pct_sample": miss.values,
        "sample_rows": len(sample_df)
    })
    return out

In [4]:
csv_files = discover_csv_files(TWITTER_ROOT, OPENASSIST_ROOT)
print(f"Discovered CSV files: {len(csv_files)}")

inventory_rows = []
schema_rows = []

for p in csv_files:
    dataset_group = "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant"
    headers = read_header(p)
    n_rows = count_data_rows(p)
    size_mb = round(p.stat().st_size / (1024 * 1024), 2)

    inventory_rows.append({
        "dataset_group": dataset_group,
        "file_path": str(p),
        "file_name": p.name,
        "size_mb": size_mb,
        "n_rows": n_rows,
        "n_columns": len(headers)
    })

    for col_idx, col in enumerate(headers):
        schema_rows.append({
            "dataset_group": dataset_group,
            "file_name": p.name,
            "column_index": col_idx,
            "column_name": col
        })

inventory_df = pd.DataFrame(inventory_rows).sort_values(["dataset_group", "file_name"])
schema_df = pd.DataFrame(schema_rows).sort_values(["dataset_group", "file_name", "column_index"])

display(inventory_df)
display(schema_df.head(40))

inventory_df.to_csv(OUT_DIR / "part1_file_inventory.csv", index=False)
schema_df.to_csv(OUT_DIR / "part1_schema_columns.csv", index=False)

print("Saved: part1_file_inventory.csv")
print("Saved: part1_schema_columns.csv")

Discovered CSV files: 4


,dataset_group,file_path,file_name,size_mb,n_rows,n_columns
2,openassistant,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,oasst1-train.csv,119.77,771472,18
3,openassistant,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,oasst1-val.csv,6.27,41102,18
0,twitter,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,sample.csv,0.02,99,7
1,twitter,B:\College\RL\AdaptiveBandit--Contextual-Bandi...,twcs.csv,492.58,3003124,7


,dataset_group,file_name,column_index,column_name
14,openassistant,oasst1-train.csv,0,message_id
15,openassistant,oasst1-train.csv,1,parent_id
16,openassistant,oasst1-train.csv,2,user_id
17,openassistant,oasst1-train.csv,3,created_date
18,openassistant,oasst1-train.csv,4,text
19,openassistant,oasst1-train.csv,5,role
20,openassistant,oasst1-train.csv,6,lang
21,openassistant,oasst1-train.csv,7,review_count
22,openassistant,oasst1-train.csv,8,review_result
23,openassistant,oasst1-train.csv,9,deleted


Saved: part1_file_inventory.csv
Saved: part1_schema_columns.csv


In [5]:
missingness_frames = []

for p in csv_files:
    try:
        miss_df = sample_missingness(p, sample_rows=50000)
        miss_df.insert(0, "file_name", p.name)
        miss_df.insert(0, "dataset_group", "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant")
        missingness_frames.append(miss_df)
    except Exception as e:
        missingness_frames.append(pd.DataFrame([{
            "dataset_group": "twitter" if str(TWITTER_ROOT) in str(p) else "openassistant",
            "file_name": p.name,
            "column": "__ERROR__",
            "missing_pct_sample": None,
            "sample_rows": 0,
            "error": str(e)
        }]))

missingness_df = pd.concat(missingness_frames, ignore_index=True)
missingness_df.to_csv(OUT_DIR / "part1_missingness_sample.csv", index=False)

summary = (
    inventory_df.groupby("dataset_group")[["n_rows", "n_columns", "size_mb"]]
    .agg({"n_rows": "sum", "n_columns": "mean", "size_mb": "sum"})
    .rename(columns={"n_rows": "total_rows", "n_columns": "avg_columns_per_file", "size_mb": "total_size_mb"})
    .reset_index()
)
summary.to_csv(OUT_DIR / "part1_dataset_summary.csv", index=False)

report_lines = [
    "# Part 1 Data Audit Report",
    "",
    "## Scope",
    "- Twitter root: " + str(TWITTER_ROOT),
    "- OpenAssistant root: " + str(OPENASSIST_ROOT),
    "",
    "## Outputs",
    "- part1_file_inventory.csv",
    "- part1_schema_columns.csv",
    "- part1_missingness_sample.csv",
    "- part1_dataset_summary.csv",
    "",
    "## Quick Summary",
]

for _, r in summary.iterrows():
    report_lines.append(
        f"- {r['dataset_group']}: rows={int(r['total_rows']):,}, avg_columns_per_file={r['avg_columns_per_file']:.2f}, size_mb={r['total_size_mb']:.2f}"
    )

(OUT_DIR / "part1_data_audit_report.md").write_text("\n".join(report_lines), encoding="utf-8")

display(summary)
display(missingness_df.head(40))
print("Saved: part1_missingness_sample.csv")
print("Saved: part1_dataset_summary.csv")
print("Saved: part1_data_audit_report.md")

,dataset_group,total_rows,avg_columns_per_file,total_size_mb
0,openassistant,812574,18.0,126.04
1,twitter,3003223,7.0,492.60


,dataset_group,file_name,column,missing_pct_sample,sample_rows
0,twitter,sample.csv,tweet_id,0.00,93
1,twitter,sample.csv,author_id,0.00,93
2,twitter,sample.csv,inbound,0.00,93
3,twitter,sample.csv,created_at,0.00,93
4,twitter,sample.csv,text,0.00,93
5,twitter,sample.csv,response_tweet_id,30.11,93
6,twitter,sample.csv,in_response_to_tweet_id,26.88,93
7,twitter,twcs.csv,tweet_id,0.00,50000
8,twitter,twcs.csv,author_id,0.00,50000
9,twitter,twcs.csv,inbound,0.00,50000


Saved: part1_missingness_sample.csv
Saved: part1_dataset_summary.csv
Saved: part1_data_audit_report.md


## Part 1 Complete

Part 1 provides the dataset inventory and schema baseline needed for Part 2 (action labeling).

Next planned implementation:
- Build an action-labeling pipeline that classifies agent turns into the 7-action space.
- Add a small annotation-ready export for inter-rater validation (kappa).

In [ ]:
# Export full labeled set
full_labeled_path = PART2_DIR / "part2_agent_turns_labeled.csv"
agent_turns_df.to_csv(full_labeled_path, index=False)

# Build a 200-row stratified annotation set for human validation (kappa workflow)
np.random.seed(42)
per_class = max(1, 200 // len(ACTION_SPACE))
annotation_parts = []

for action in ACTION_SPACE:
    pool = agent_turns_df[agent_turns_df["action_label"] == action]
    if pool.empty:
        continue
    take = min(per_class, len(pool))
    annotation_parts.append(pool.sample(n=take, random_state=42))

annotation_df = pd.concat(annotation_parts, ignore_index=True)

if len(annotation_df) < 200:
    extra_needed = 200 - len(annotation_df)
    remaining = agent_turns_df.drop(annotation_df.index, errors="ignore")
    if len(remaining) > 0:
        annotation_df = pd.concat(
            [annotation_df, remaining.sample(n=min(extra_needed, len(remaining)), random_state=42)],
            ignore_index=True,
        )

annotation_df = annotation_df.head(200).copy()
annotation_df.insert(0, "annotation_id", [f"ann_{i:04d}" for i in range(1, len(annotation_df) + 1)])
annotation_df["human_label"] = ""
annotation_df["human_notes"] = ""

annotation_export_cols = [
    "annotation_id",
    "source_file",
    "dataset_group",
    "agent_text",
    "action_label",
    "action_id",
    "action_confidence",
    "label_reason",
    "human_label",
    "human_notes",
]

annotation_path = PART2_DIR / "part2_annotation_sample_200.csv"
annotation_df[annotation_export_cols].to_csv(annotation_path, index=False)

label_dist = agent_turns_df["action_label"].value_counts().rename_axis("action_label").reset_index(name="count")
label_dist["pct"] = (label_dist["count"] / label_dist["count"].sum() * 100).round(2)
label_dist_path = PART2_DIR / "part2_label_distribution.csv"
label_dist.to_csv(label_dist_path, index=False)

prompt_template = {
    "system": "You are an expert customer support analyst.",
    "instruction": "Classify the agent utterance into exactly one of the 7 actions.",
    "actions": ACTION_SPACE,
    "output_format": {
        "action_label": "string",
        "confidence": "0.0-1.0",
        "reasoning": "short rationale"
    },
}

with (PART2_DIR / "part2_llm_label_prompt_template.json").open("w", encoding="utf-8") as f:
    json.dump(prompt_template, f, indent=2)

summary_rows = [
    {"metric": "total_labeled_turns", "value": int(len(agent_turns_df))},
    {"metric": "annotation_sample_size", "value": int(len(annotation_df))},
    {"metric": "num_action_classes_present", "value": int(agent_turns_df['action_label'].nunique())},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PART2_DIR / "part2_summary_metrics.csv", index=False)

display(label_dist)
print("Saved:", full_labeled_path)
print("Saved:", annotation_path)
print("Saved:", label_dist_path)
print("Saved:", PART2_DIR / "part2_summary_metrics.csv")
print("Saved:", PART2_DIR / "part2_llm_label_prompt_template.json")

In [ ]:
def extract_agent_turns_from_csv(path, max_rows=300000, chunk_size=100000):
    """Extract candidate agent turns with robust fallbacks across unknown schemas."""
    records = []
    source_name = path.name
    is_twitter = "twitter" in str(path).lower() or "twcs" in source_name.lower()

    try:
        for chunk in pd.read_csv(path, chunksize=chunk_size, low_memory=False):
            if len(records) >= max_rows:
                break

            cols = set(chunk.columns)

            # Case 1: Twitter-like schema with inbound flag.
            if {"inbound", "text"}.issubset(cols):
                sub = chunk[chunk["inbound"].astype(str).str.lower().isin(["false", "0"])].copy()
                if sub.empty:
                    continue
                keep_cols = [c for c in ["tweet_id", "author_id", "created_at", "text", "in_response_to_tweet_id", "response_tweet_id"] if c in sub.columns]
                sub = sub[keep_cols].copy()
                sub["source_file"] = source_name
                sub["dataset_group"] = "twitter" if is_twitter else "openassistant"
                sub["speaker_proxy"] = "agent"
                sub = sub.rename(columns={"text": "agent_text"})
                records.extend(sub.to_dict("records"))
                continue

            # Case 2: role/speaker columns for assistant turns.
            role_col = None
            for c in ["role", "speaker", "from", "author_role"]:
                if c in cols:
                    role_col = c
                    break

            text_col = None
            for c in ["text", "message", "utterance", "content"]:
                if c in cols:
                    text_col = c
                    break

            if role_col and text_col:
                mask = chunk[role_col].astype(str).str.lower().isin(["assistant", "agent", "support", "system"])
                sub = chunk.loc[mask, [role_col, text_col]].copy()
                if sub.empty:
                    continue
                sub["source_file"] = source_name
                sub["dataset_group"] = "openassistant" if "openassistant" in str(path).lower() else "twitter"
                sub["speaker_proxy"] = "agent"
                sub = sub.rename(columns={text_col: "agent_text"})
                records.extend(sub.to_dict("records"))

            if len(records) >= max_rows:
                break

    except Exception as e:
        print(f"Skipping {source_name} due to read error: {e}")

    if not records:
        return pd.DataFrame(columns=["source_file", "dataset_group", "speaker_proxy", "agent_text"])

    out = pd.DataFrame(records).head(max_rows)
    out["agent_text"] = out["agent_text"].map(clean_text)
    out = out[out["agent_text"].str.len() > 0].copy()
    return out


agent_turns_frames = []
for p in csv_files:
    extracted = extract_agent_turns_from_csv(p, max_rows=200000 if "twcs" in p.name.lower() else 50000)
    if not extracted.empty:
        agent_turns_frames.append(extracted)

if not agent_turns_frames:
    raise RuntimeError("No agent-turn candidates found. Check source schema assumptions.")

agent_turns_df = pd.concat(agent_turns_frames, ignore_index=True)
agent_turns_df = agent_turns_df.drop_duplicates(subset=["source_file", "agent_text"]).reset_index(drop=True)

labels = agent_turns_df["agent_text"].map(heuristic_action_label)
agent_turns_df["action_label"] = labels.map(lambda x: x[0])
agent_turns_df["action_id"] = agent_turns_df["action_label"].map(ACTION_TO_ID)
agent_turns_df["action_confidence"] = labels.map(lambda x: x[1])
agent_turns_df["label_reason"] = labels.map(lambda x: x[2])

print(f"Agent turns labeled: {len(agent_turns_df):,}")
agent_turns_df[["source_file", "dataset_group", "agent_text", "action_label", "action_confidence"]].head(10)

In [ ]:
import re
import numpy as np

PART2_DIR = ROOT / "Simulation" / "artifacts" / "part2_action_labeling"
PART2_DIR.mkdir(parents=True, exist_ok=True)

ACTION_SPACE = [
    "Ask_for_Information",
    "Provide_Solution",
    "Affective_Repair",
    "Escalate_to_Human",
    "Close_with_Feedback",
    "Proactive_Update",
    "Set_Expectation",
]

ACTION_TO_ID = {a: i for i, a in enumerate(ACTION_SPACE)}

ACTION_PATTERNS = {
    "Ask_for_Information": [
        r"\bcan you provide\b", r"\bplease provide\b", r"\bcould you share\b",
        r"\bwhat is your\b", r"\bmay i have\b", r"\border number\b", r"\baccount\b"
    ],
    "Provide_Solution": [
        r"\bplease try\b", r"\byou can\b", r"\bhere is how\b", r"\bsteps?\b",
        r"\bresolved\b", r"\bfix\b", r"\bsolution\b", r"\bupdate your app\b"
    ],
    "Affective_Repair": [
        r"\bsorry\b", r"\bapolog\w*\b", r"\bunderstand how\b", r"\bfrustrat\w*\b",
        r"\bwe understand\b", r"\bthanks for your patience\b"
    ],
    "Escalate_to_Human": [
        r"\bescalat\w*\b", r"\bsupervisor\b", r"\bspecialist\b", r"\bhuman agent\b",
        r"\bforward this to\b", r"\bcontact support team\b"
    ],
    "Close_with_Feedback": [
        r"\banything else\b", r"\bhappy to help\b", r"\bglad to help\b",
        r"\brate your experience\b", r"\bthank you\b", r"\bhave a great day\b"
    ],
    "Proactive_Update": [
        r"\bquick update\b", r"\bstatus update\b", r"\bwe are still working\b",
        r"\bkeeping you posted\b", r"\bno action needed yet\b"
    ],
    "Set_Expectation": [
        r"\bwithin \d+\s?(minutes?|hours?|days?)\b", r"\bby end of day\b", r"\bexpect\b",
        r"\bnext steps\b", r"\btimeline\b", r"\bETA\b"
    ],
}


def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def heuristic_action_label(text):
    t = clean_text(text).lower()
    if not t:
        return "Provide_Solution", 0.2, "empty_or_missing_text_default"

    scores = {a: 0 for a in ACTION_SPACE}
    reasons = []

    for action, pats in ACTION_PATTERNS.items():
        for pat in pats:
            if re.search(pat, t):
                scores[action] += 1
                reasons.append(f"{action}:{pat}")

    # Tie-break preference roughly aligned with operational criticality.
    tie_order = [
        "Escalate_to_Human",
        "Affective_Repair",
        "Ask_for_Information",
        "Set_Expectation",
        "Proactive_Update",
        "Provide_Solution",
        "Close_with_Feedback",
    ]

    best_score = max(scores.values())
    if best_score == 0:
        label = "Provide_Solution"
        confidence = 0.35
        reason = "no_pattern_match_default"
    else:
        top = [a for a, s in scores.items() if s == best_score]
        label = sorted(top, key=lambda x: tie_order.index(x))[0]
        confidence = min(0.5 + 0.12 * best_score, 0.95)
        reason = ";".join(reasons[:5])

    return label, confidence, reason

## Part 2: Action Labeling Pipeline (Implemented)

This part builds a practical action-labeling workflow for agent turns.

What this section does:
- Defines the 7-action taxonomy from the implementation plan
- Extracts likely agent turns from available CSV files
- Applies a deterministic heuristic labeler (baseline bootstrap)
- Exports a full labeled dataset and a 200-row annotation set for human validation

Output folder:
- Simulation/artifacts/part2_action_labeling